# Week 3 : Imbalanced Classification and LightGBM Modeling

**Day 1 - Confirm the imbalance problem**

Goal for today: load the fused dataset from Week 2, measure exactly how rare machine failures are, installing the libraries needed and see concretely why accuracy is the wrong metric for this problem.

In [ ]:
import pandas as pd
import numpy as np

import lightgbm as lgb
import imblearn

print("lightgbm:", lgb.__version__)
print("imbalanced-learn:", imblearn.__version__)

In [ ]:
DATA_PATH = "ai4i_fused_week2.csv"  # your Week 2 output

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## Step 1 - Measure exactly how rare failures are

The brief states failures are under 2% of the data - let's verify that on your actual fused dataset rather than assuming.

In [ ]:
target = "Machine failure"

counts = df[target].value_counts()
pct = df[target].value_counts(normalize=True) * 100

print(counts)
print()
print(pct.round(3))

## Step 2 - Why accuracy will lie to you

Before touching LightGBM, let's prove the point with the simplest possible "model": one that always predicts "no failure," no matter what. If this dummy model still scores high on accuracy, we know that accuracy on these models is compatible with further study

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score

y = df[target]

y_train, y_test = train_test_split(y, test_size=0.2, stratify=y, random_state=42)

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(np.zeros((len(y_train), 1)), y_train)
y_pred = dummy.predict(np.zeros((len(y_test), 1)))

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall (failures caught): {recall_score(y_test, y_pred):.4f}")
print(f"F1 score: {f1_score(y_test, y_pred):.4f}")

## Takeaway for Day 1

A model that never predicts a failure can score over 95% accuracy while catching **zero** real failures - exactly the opposite of what a predictive maintenance system needs to do. For the rest of this week, judge every model by **recall**, **F1**, and **ROC-AUC / PR-AUC**, never accuracy alone.